# HypeGuard — Notebook 02: Model Training
**Run cells top to bottom. Do not skip any cell.**

**What this notebook does:**
1. Fetches real data for 11 tickers (takes ~3 minutes)
2. Runs EDA — proves features separate HYPE from ORGANIC
3. Trains Isolation Forest (anomaly detection, unsupervised)
4. Trains Random Forest (hype classifier, supervised via pseudo-labels)
5. Exports `random_forest.pkl` and `isolation_forest.pkl` to `../backend/models/`

**Deliverables:** 2 `.pkl` files + 3 charts saved to `../data/`

In [1]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — IMPORTS & STYLE
# ═══════════════════════════════════════════════════════════
import sys, os, pickle, warnings
sys.path.append('../backend/src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.ensemble        import IsolationForest, RandomForestClassifier
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.model_selection import LeaveOneOut, cross_val_score, StratifiedKFold
from sklearn.metrics         import classification_report, confusion_matrix

from scraper  import collect_all
from features import build_feature_vector

# ── Plot style ─────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'text.color':       'white',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'axes.edgecolor':   '#444',
    'grid.color':       '#333',
    'font.family':      'monospace',
})

RED    = '#ef4444'
GREEN  = '#22c55e'
BLUE   = '#3b82f6'
AMBER  = '#f59e0b'
PURPLE = '#a855f7'
GRAY   = '#6b7280'

# ── Feature order — FROZEN. Do not change. ─────────────────
FEATURE_ORDER = [
    'rvol', 'volume_zscore', 'vol_price_divergence', 'vol_spike_days', 'vol_trend_slope_norm',
    'log_return_1d', 'price_vs_sma20', 'rsi_14', 'bb_width', 'gap_open', 'range_expansion',
    'buzz_density', 'extreme_language_ratio', 'moderate_hype_ratio',
    'bearish_ratio', 'source_diversity', 'headline_similarity',
    'catalyst_flag', 'hype_without_catalyst', 'news_volume_sync', 'silent_spike'
]

LABEL_MAP     = {'ORGANIC': 0, 'HYPE': 1, 'INSTITUTIONAL': 2, 'NEUTRAL': 3}
LABEL_MAP_INV = {v: k for k, v in LABEL_MAP.items()}
LABEL_COLORS  = {'ORGANIC': GREEN, 'HYPE': RED, 'INSTITUTIONAL': BLUE, 'NEUTRAL': GRAY}

os.makedirs('../backend/models', exist_ok=True)
os.makedirs('../data',           exist_ok=True)

print('✓ Setup complete')
print(f'  Feature count : {len(FEATURE_ORDER)}')
print(f'  Label classes : {list(LABEL_MAP.keys())}')

✓ Setup complete
  Feature count : 21
  Label classes : ['ORGANIC', 'HYPE', 'INSTITUTIONAL', 'NEUTRAL']


In [5]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — DATA COLLECTION
# Takes ~3 minutes. Each ticker = 1 API call to yfinance + Google News.
# ═══════════════════════════════════════════════════════════

# Tickers chosen for MAXIMUM contrast across labels.
# Do NOT change this list — it's calibrated for the pseudo-labeling thresholds.
TICKERS_CONFIG = {
    # ── Expected HYPE / PUMP ───────────────────────────────
    'GME':   {'days': 60, 'note': 'Classic meme pump'},
    'AMC':   {'days': 60, 'note': 'Meme hype 2021'},
    'BBBY':  {'days': 60, 'note': 'Pre-bankruptcy pump 2022'},
    'CLOV':  {'days': 60, 'note': 'WSB meme stock'},
    # ── Expected ORGANIC ──────────────────────────────────
    'NVDA':  {'days': 60, 'note': 'AI-driven organic growth'},
    'AAPL':  {'days': 60, 'note': 'Steady blue chip'},
    'MSFT':  {'days': 60, 'note': 'Steady blue chip'},
    'GOOGL': {'days': 60, 'note': 'Large cap organic'},
    # ── Expected NEUTRAL / INSTITUTIONAL ──────────────────
    'JPM':   {'days': 60, 'note': 'Institutional banking'},
    'BRK-B': {'days': 60, 'note': 'Berkshire — zero hype'},
    'XOM':   {'days': 60, 'note': 'Energy — low social noise'},
}

print(f'Collecting data for {len(TICKERS_CONFIG)} tickers...')
print('─' * 55)

rows = []
failed = []

for ticker, cfg in TICKERS_CONFIG.items():
    try:
        raw = collect_all(ticker, days=cfg['days'])
        dq  = raw['data_quality']

        if not dq['has_price_data']:
            print(f'  ⚠  {ticker:6s} — no price data, skipping')
            failed.append(ticker)
            continue

        fv   = build_feature_vector(raw)
        flat = fv['flat_features'].copy()

        # Add metadata columns (excluded from model, kept for EDA)
        flat['ticker']         = ticker
        flat['pseudo_label']   = fv['cross_features']['pseudo_label']
        flat['hype_score_raw'] = fv['cross_features']['hype_score_raw']
        flat['news_count']     = dq['news_count']
        flat['note']           = cfg['note']

        rows.append(flat)
        label  = flat['pseudo_label']
        score  = flat['hype_score_raw']
        icon   = '🔴' if label in ('HYPE','PUMP_ALERT') else '🟢' if label == 'ORGANIC' else '🔵'
        print(f'  {icon} {ticker:6s} | label={label:13s} | score={score:.2f} | news={dq["news_count"]}')

    except Exception as e:
        print(f'  ✗  {ticker:6s} — ERROR: {e}')
        failed.append(ticker)

df = pd.DataFrame(rows)

print('─' * 55)
print(f'\nDataset shape : {df.shape}')
print(f'Failed tickers: {failed if failed else "None"}')
print(f'\nLabel distribution:')
print(df['pseudo_label'].value_counts().to_string())

df.to_csv('../data/training_data.csv', index=False)
print('\n✓ Saved: data/training_data.csv')

INFO | ==================================================
INFO |   HypeGuard Data Collection: GME
INFO | ==================================================
INFO | Fetching price data for GME (60 days)...
INFO |   ✓ 43 trading days fetched for GME
INFO | Fetching news for GME from Google News RSS...


───────────────────────────────────────────────────────


INFO |   ✓ 30 news articles fetched for GME
WARNING | Could not fetch earnings dates for GME: 'dict' object has no attribute 'empty'
INFO | 
Data Quality Report for GME:
INFO |   has_price_data: True
INFO |   has_news: True
INFO |   has_earnings: False
INFO |   price_rows: 43
INFO |   news_count: 30
INFO | Building feature vector for GME...
INFO |   ✓ Feature vector built: 21 features, label=NEUTRAL
INFO | ==================================================
INFO |   HypeGuard Data Collection: AMC
INFO | ==================================================
INFO | Fetching price data for AMC (60 days)...
INFO |   ✓ 43 trading days fetched for AMC
INFO | Fetching news for AMC from Google News RSS...


  🔵 GME    | label=NEUTRAL       | score=0.09 | news=30


INFO |   ✓ 30 news articles fetched for AMC
WARNING | Could not fetch earnings dates for AMC: 'dict' object has no attribute 'empty'
INFO | 
Data Quality Report for AMC:
INFO |   has_price_data: True
INFO |   has_news: True
INFO |   has_earnings: False
INFO |   price_rows: 43
INFO |   news_count: 30
INFO | Building feature vector for AMC...
INFO |   ✓ Feature vector built: 21 features, label=NEUTRAL
INFO | ==================================================
INFO |   HypeGuard Data Collection: BBBY
INFO | ==================================================
INFO | Fetching price data for BBBY (60 days)...
INFO |   ✓ 43 trading days fetched for BBBY
INFO | Fetching news for BBBY from Google News RSS...


...

  🔵 JPM    | label=NEUTRAL       | score=0.10 | news=30


INFO |   ✓ 30 news articles fetched for BRK-B
WARNING | Could not fetch earnings dates for BRK-B: 'dict' object has no attribute 'empty'
INFO | 
Data Quality Report for BRK-B:
INFO |   has_price_data: True
INFO |   has_news: True
INFO |   has_earnings: False
INFO |   price_rows: 43
INFO |   news_count: 30
INFO | Building feature vector for BRK-B...
INFO |   ✓ Feature vector built: 21 features, label=NEUTRAL
INFO | ==================================================
INFO |   HypeGuard Data Collection: XOM
INFO | ==================================================
INFO | Fetching price data for XOM (60 days)...
INFO |   ✓ 43 trading days fetched for XOM
INFO | Fetching news for XOM from Google News RSS...


  🔵 BRK-B  | label=NEUTRAL       | score=0.09 | news=30


INFO |   ✓ 30 news articles fetched for XOM
WARNING | Could not fetch earnings dates for XOM: 'dict' object has no attribute 'empty'
INFO | 
Data Quality Report for XOM:
INFO |   has_price_data: True
INFO |   has_news: True
INFO |   has_earnings: False
INFO |   price_rows: 43
INFO |   news_count: 30
INFO | Building feature vector for XOM...
INFO |   ✓ Feature vector built: 21 features, label=NEUTRAL


  🔵 XOM    | label=NEUTRAL       | score=0.25 | news=30
───────────────────────────────────────────────────────

Dataset shape : (11, 26)
Failed tickers: None

Label distribution:
pseudo_label
NEUTRAL    11

✓ Saved: data/training_data.csv


In [7]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — SANITY CHECK: Did pseudo-labeling work?
# GME/AMC/BBBY must NOT all be 'NEUTRAL'. If they are, 
# features.py thresholds need adjustment.
# ═══════════════════════════════════════════════════════════

print('Label assignment check:')
print('─' * 45)
summary = df[['ticker', 'pseudo_label', 'hype_score_raw', 'rvol', 'rsi_14']].copy()
summary['hype_pct'] = (summary['hype_score_raw'] * 100).round(1)
summary = summary.sort_values('hype_score_raw', ascending=False)

for _, row in summary.iterrows():
    icon = '🔴' if row['pseudo_label'] in ('HYPE','PUMP_ALERT') else \
           '🟢' if row['pseudo_label'] == 'ORGANIC' else '🔵'
    print(f'  {icon} {row["ticker"]:6s} | {row["pseudo_label"]:13s} | '
          f'hype={row["hype_pct"]:5.1f}% | RVOL={row["rvol"]:.2f} | RSI={row["rsi_14"]:.0f}')

# Automatic check
hype_tickers    = set(df[df['pseudo_label'].isin(['HYPE','PUMP_ALERT'])]['ticker'])
organic_tickers = set(df[df['pseudo_label'] == 'ORGANIC']['ticker'])

print('\n' + '─' * 45)
if not hype_tickers:
    print('⛔ WARNING: No HYPE labels generated!')
    print('   → In features.py, lower RVOL_SPIKE_THRESHOLD from 2.5 to 1.8')
    print('   → Or lower ZSCORE_SPIKE_THRESHOLD from 2.0 to 1.5')
elif not organic_tickers:
    print('⛔ WARNING: No ORGANIC labels generated!')
    print('   → Check if stable tickers have unexpected volume spikes today')
else:
    print(f'✓ HYPE tickers    : {hype_tickers}')
    print(f'✓ ORGANIC tickers : {organic_tickers}')
    print('\nLabel balance looks good. Proceed to Cell 4.')

Label assignment check:
─────────────────────────────────────────────
  🔵 XOM    | NEUTRAL       | hype= 24.7% | RVOL=1.24 | RSI=82
  🔵 BBBY   | NEUTRAL       | hype= 14.2% | RVOL=1.35 | RSI=32
  🔵 CLOV   | NEUTRAL       | hype= 12.3% | RVOL=0.66 | RSI=14
  🔵 GOOGL  | NEUTRAL       | hype= 12.3% | RVOL=1.21 | RSI=22
  🔵 AMC    | NEUTRAL       | hype= 11.9% | RVOL=0.61 | RSI=24
  🔵 MSFT   | NEUTRAL       | hype= 11.6% | RVOL=1.15 | RSI=9
  🔵 AAPL   | NEUTRAL       | hype= 11.2% | RVOL=1.18 | RSI=32
  🔵 JPM    | NEUTRAL       | hype= 10.5% | RVOL=0.90 | RSI=40
  🔵 NVDA   | NEUTRAL       | hype= 10.0% | RVOL=1.08 | RSI=31
  🔵 BRK-B  | NEUTRAL       | hype=  9.5% | RVOL=1.03 | RSI=8
  🔵 GME    | NEUTRAL       | hype=  8.9% | RVOL=0.92 | RSI=23

─────────────────────────────────────────────
⛔ WARNING: No HYPE labels generated!
   → In features.py, lower RVOL_SPIKE_THRESHOLD from 2.5 to 1.8
   → Or lower ZSCORE_SPIKE_THRESHOLD from 2.0 to 1.5


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — EDA CHART A: Feature Distributions by Label
# This is the most important chart for your report.
# It visually proves features SEPARATE the classes.
# ═══════════════════════════════════════════════════════════

KEY_FEATURES = [
    ('rvol',                    'Relative Volume (RVOL)',       'Higher = more anomalous volume'),
    ('volume_zscore',           'Volume Z-Score',               'Std deviations above 20-day avg'),
    ('rsi_14',                  'RSI (14-day)',                  '>75 = overbought'),
    ('extreme_language_ratio',  'Extreme Hype Language',        'moon/rocket/squeeze keywords'),
    ('source_diversity',        'News Source Diversity',        'Low = coordinated campaign'),
    ('headline_similarity',     'Headline Similarity Score',    'High = copy-paste journalism'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('EDA: Feature Distributions by Label\n'
             '(Shows that features meaningfully separate HYPE from ORGANIC)',
             fontsize=13, y=1.01)

for ax, (feat, title, subtitle) in zip(axes.flat, KEY_FEATURES):
    for label in df['pseudo_label'].unique():
        subset = df[df['pseudo_label'] == label][feat].dropna()
        color  = LABEL_COLORS.get(label, GRAY)
        if len(subset) > 0:
            # Jitter x for visibility
            jitter = np.random.uniform(-0.08, 0.08, len(subset))
            x_pos  = [list(df['pseudo_label'].unique()).index(label)] * len(subset)
            ax.scatter(np.array(x_pos) + jitter, subset,
                       color=color, alpha=0.85, s=120, zorder=3,
                       edgecolors='white', linewidths=0.5, label=label)
            # Mean line
            ax.hlines(subset.mean(),
                      x_pos[0] + jitter.min() - 0.05,
                      x_pos[0] + jitter.max() + 0.05,
                      color=color, linewidth=2.5, alpha=0.6)

    labels_present = df['pseudo_label'].unique().tolist()
    ax.set_xticks(range(len(labels_present)))
    ax.set_xticklabels(labels_present, rotation=30, ha='right', fontsize=9)
    ax.set_title(f'{title}\n{subtitle}', fontsize=10, pad=8)
    ax.grid(axis='y', alpha=0.3)

# Legend
handles = [mpatches.Patch(color=c, label=l) for l, c in LABEL_COLORS.items()]
fig.legend(handles=handles, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.03), fontsize=10)

plt.tight_layout()
plt.savefig('../data/eda_feature_distributions.png',
            dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✓ Saved: data/eda_feature_distributions.png')
print('\nKey insight to mention in report:')
print('  RVOL and extreme_language_ratio should be visibly higher for HYPE vs ORGANIC')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — EDA CHART B: Feature Variance Analysis
# Low-variance features = useless to model. Drop them.
# ═══════════════════════════════════════════════════════════

feature_df = df[FEATURE_ORDER].copy().fillna(0)
variances  = feature_df.var().sort_values(ascending=True)

LOW_VAR_THRESHOLD  = 0.001
low_var_features   = variances[variances < LOW_VAR_THRESHOLD].index.tolist()
good_features      = variances[variances >= LOW_VAR_THRESHOLD].index.tolist()

bar_colors = [RED if v < LOW_VAR_THRESHOLD else BLUE
              for v in variances.values]

fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(variances.index, variances.values, color=bar_colors, alpha=0.85)
ax.axvline(LOW_VAR_THRESHOLD, color=AMBER, linestyle='--',
           linewidth=1.5, label=f'Drop threshold ({LOW_VAR_THRESHOLD})')
ax.set_xlabel('Variance across tickers')
ax.set_title('Feature Variance — Pre-Training Importance Proxy\n'
             'Red bars = near-zero variance = dropped from model',
             fontsize=12)
ax.legend()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/eda_feature_variance.png',
            dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

print(f'Total features     : {len(FEATURE_ORDER)}')
print(f'Low-var (dropped)  : {low_var_features}')
print(f'Good features kept : {len(good_features)}')
print('✓ Saved: data/eda_feature_variance.png')

# Update FEATURE_ORDER to drop low-variance ones if any
FEATURES_FINAL = [f for f in FEATURE_ORDER if f not in low_var_features]
print(f'\nFINAL_FEATURES ({len(FEATURES_FINAL)}): {FEATURES_FINAL}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — EDA CHART C: Correlation Heatmap
# Highly correlated pairs = redundant features. Good to know before training.
# ═══════════════════════════════════════════════════════════

corr_matrix = df[FEATURES_FINAL].fillna(0).corr()

fig, ax = plt.subplots(figsize=(14, 11))
im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8)

n = len(corr_matrix)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(corr_matrix.columns, fontsize=8)

# Annotate only if small enough
if n <= 21:
    for i in range(n):
        for j in range(n):
            val = corr_matrix.iloc[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=6, color='white' if abs(val) > 0.6 else '#888')

ax.set_title('Feature Correlation Matrix\n'
             'Dark red/blue = highly correlated pairs (redundant information)',
             fontsize=11)
plt.tight_layout()
plt.savefig('../data/eda_correlation_heatmap.png',
            dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

# Find highly correlated pairs (|r| > 0.8)
high_corr_pairs = []
for i in range(n):
    for j in range(i+1, n):
        r = abs(corr_matrix.iloc[i, j])
        if r > 0.8:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], round(r, 3)))

if high_corr_pairs:
    print('Highly correlated pairs (|r|>0.8) — both kept, RF handles this naturally:')
    for a, b, r in high_corr_pairs:
        print(f'  {a:30s} ↔ {b:30s}  r={r}')
else:
    print('No highly correlated pairs found — all features are independent enough.')
print('\n✓ Saved: data/eda_correlation_heatmap.png')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — TRAIN ISOLATION FOREST
# Unsupervised anomaly detector. No labels needed.
# Detects "weird" price-volume behavior.
# ═══════════════════════════════════════════════════════════

X_raw = df[FEATURES_FINAL].fillna(0).values

# Scale (Isolation Forest uses distances — scaling matters)
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

iso = IsolationForest(
    n_estimators  = 300,
    contamination = 0.25,    # ~25% of current stocks expected to be anomalous
    max_features  = 0.8,     # use 80% features per tree (reduces overfitting)
    random_state  = 42,
    n_jobs        = -1,
)
iso.fit(X_scaled)

# Score: more negative = more anomalous
raw_scores = iso.score_samples(X_scaled)
# Normalize to 0–1: 0 = normal, 1 = extreme outlier
anomaly_scores_norm = np.clip(1 - (raw_scores + 0.5), 0, 1)

df['anomaly_score'] = anomaly_scores_norm
df['is_anomaly']    = iso.predict(X_scaled) == -1  # True = outlier

print('Isolation Forest — Results:')
print('─' * 50)
iso_df = df[['ticker', 'pseudo_label', 'anomaly_score', 'is_anomaly']].copy()
iso_df = iso_df.sort_values('anomaly_score', ascending=False)
for _, row in iso_df.iterrows():
    icon = '🔴' if row['is_anomaly'] else '🟢'
    bar  = '█' * int(row['anomaly_score'] * 20)
    print(f'  {icon} {row["ticker"]:6s} | {row["pseudo_label"]:13s} | '
          f'score={row["anomaly_score"]:.3f} |{bar}')

print(f'\n  Flagged anomalies : {df["is_anomaly"].sum()} / {len(df)}')

# Validation: HYPE tickers should score higher than ORGANIC
hype_mean    = df[df['pseudo_label'].isin(['HYPE','PUMP_ALERT'])]['anomaly_score'].mean()
organic_mean = df[df['pseudo_label'] == 'ORGANIC']['anomaly_score'].mean()
print(f'\n  HYPE avg anomaly score    : {hype_mean:.3f}')
print(f'  ORGANIC avg anomaly score : {organic_mean:.3f}')

if hype_mean > organic_mean:
    print('  ✅ Isolation Forest is working correctly.')
else:
    print('  ⚠️  HYPE tickers not scoring higher. Try contamination=0.3')

# Save model bundle — includes scaler and feature order
iso_bundle = {
    'model':         iso,
    'scaler':        scaler,
    'feature_order': FEATURES_FINAL,
}
with open('../backend/models/isolation_forest.pkl', 'wb') as f:
    pickle.dump(iso_bundle, f)
print('\n✓ Saved: backend/models/isolation_forest.pkl')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — PREPARE TRAINING MATRIX FOR RANDOM FOREST
# ═══════════════════════════════════════════════════════════

# Encode labels to integers
df['label_encoded'] = df['pseudo_label'].map(LABEL_MAP).fillna(3).astype(int)

X = df[FEATURES_FINAL].fillna(0).values
y = df['label_encoded'].values
tickers_list = df['ticker'].tolist()

print('Training Matrix:')
print(f'  Samples (tickers)  : {X.shape[0]}')
print(f'  Features           : {X.shape[1]}')
print(f'  Feature list       : {FEATURES_FINAL}')
print()
print('Class Distribution:')
unique, counts = np.unique(y, return_counts=True)
for cls, cnt in zip(unique, counts):
    label_name = LABEL_MAP_INV[cls]
    bar = '█' * cnt
    print(f'  {label_name:13s} (class {cls}): {cnt} samples  {bar}')

# Check: need at least 2 samples of each class that appears
if len(unique) < 2:
    print('\n⛔ Only 1 class found. Model cannot train. Fix pseudo-labeling first.')
else:
    print(f'\n✓ {len(unique)} classes found. Ready to train Random Forest.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — TRAIN RANDOM FOREST
# This is the main prediction model.
# Output: predict_proba → hype probability 0–100%
# ═══════════════════════════════════════════════════════════

rf = RandomForestClassifier(
    n_estimators   = 500,
    max_depth      = 4,        # shallow = less overfitting on small dataset
    min_samples_leaf = 1,
    max_features   = 'sqrt',   # sqrt(n_features) per split = reduces variance
    class_weight   = 'balanced', # handles class imbalance automatically
    random_state   = 42,
    n_jobs         = -1,
)
rf.fit(X, y)

# ── Training accuracy (sanity check — not the real metric) ─
train_preds = rf.predict(X)
train_acc   = (train_preds == y).mean()
print(f'Training accuracy  : {train_acc:.1%}  (expected high — small dataset)')

# ── Leave-One-Out Cross-Validation ─────────────────────────
# LOO is the right CV for small datasets (n < 30)
# Each ticker is the test set once, rest train
if len(X) >= 4:
    loo       = LeaveOneOut()
    loo_preds = []
    loo_true  = []

    for train_idx, test_idx in loo.split(X):
        rf_loo = RandomForestClassifier(
            n_estimators=300, max_depth=4, min_samples_leaf=1,
            max_features='sqrt', class_weight='balanced',
            random_state=42, n_jobs=-1
        )
        rf_loo.fit(X[train_idx], y[train_idx])
        pred = rf_loo.predict(X[test_idx])[0]
        loo_preds.append(pred)
        loo_true.append(y[test_idx][0])

    loo_acc = (np.array(loo_preds) == np.array(loo_true)).mean()
    print(f'LOO-CV accuracy    : {loo_acc:.1%}  ← USE THIS for the report')
    print(f'  ({len(X)} tickers, each used as test set once)')

    # Per-ticker LOO report
    print('\nPer-ticker LOO predictions:')
    print('─' * 55)
    for ticker, true_cls, pred_cls in zip(tickers_list, loo_true, loo_preds):
        true_label = LABEL_MAP_INV[true_cls]
        pred_label = LABEL_MAP_INV[pred_cls]
        match = '✅' if true_cls == pred_cls else '❌'
        print(f'  {match} {ticker:6s} | true={true_label:13s} | predicted={pred_label}')
else:
    print('Not enough samples for LOO CV (need >= 4)')

# Save the main model (trained on ALL data)
rf_bundle = {
    'model':         rf,
    'feature_order': FEATURES_FINAL,
    'label_map':     LABEL_MAP,
    'label_map_inv': LABEL_MAP_INV,
    'loo_accuracy':  loo_acc if len(X) >= 4 else None,
}
with open('../backend/models/random_forest.pkl', 'wb') as f:
    pickle.dump(rf_bundle, f)
print('\n✓ Saved: backend/models/random_forest.pkl')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — FEATURE IMPORTANCE CHART
# Shows WHICH features the model is actually using.
# This is your strongest slide in the presentation.
# ═══════════════════════════════════════════════════════════

importances = pd.Series(rf.feature_importances_, index=FEATURES_FINAL)
importances = importances.sort_values(ascending=True)  # ascending for horizontal bar

# Color by feature group
def feature_color(name):
    volume_feats = ['rvol','volume_zscore','vol_price_divergence','vol_spike_days','vol_trend_slope_norm']
    price_feats  = ['log_return_1d','price_vs_sma20','rsi_14','bb_width','gap_open','range_expansion']
    news_feats   = ['buzz_density','extreme_language_ratio','moderate_hype_ratio',
                    'bearish_ratio','source_diversity','headline_similarity']
    if name in volume_feats: return AMBER
    if name in price_feats:  return BLUE
    if name in news_feats:   return PURPLE
    return GRAY  # cross features

bar_colors = [feature_color(f) for f in importances.index]

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(importances.index, importances.values, color=bar_colors, alpha=0.88)

# Value labels
for bar, val in zip(bars, importances.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8, color='white')

# Legend
legend_patches = [
    mpatches.Patch(color=AMBER,  label='Volume Features'),
    mpatches.Patch(color=BLUE,   label='Price Features'),
    mpatches.Patch(color=PURPLE, label='News/Sentiment Features'),
    mpatches.Patch(color=GRAY,   label='Cross Features'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
ax.set_xlabel('Gini Importance')
ax.set_title('Random Forest — Feature Importances\n'
             'Which features drive the Hype Score most?', fontsize=12)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/model_feature_importance.png',
            dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

print('Top 5 features (use these as talking points):')
for feat, imp in importances.sort_values(ascending=False).head(5).items():
    group = 'Volume' if feature_color(feat)==AMBER else \
            'Price' if feature_color(feat)==BLUE else \
            'News' if feature_color(feat)==PURPLE else 'Cross'
    print(f'  [{group:6s}] {feat:35s}: {imp:.4f}')

print('✓ Saved: data/model_feature_importance.png')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — PREDICT_PROBA SHOWCASE
# This shows exactly what the API will use:
# a probability score PER class, not just a label.
# ═══════════════════════════════════════════════════════════

probas = rf.predict_proba(X)

# The model might not have all 4 classes if some are missing from training data
# rf.classes_ tells us which classes it knows
class_names = [LABEL_MAP_INV[c] for c in rf.classes_]
print(f'Model classes: {class_names}  (indices: {rf.classes_.tolist()})')
print()

# Hype class index — find where HYPE is in rf.classes_
hype_class_idx = list(rf.classes_).index(LABEL_MAP['HYPE']) \
    if LABEL_MAP['HYPE'] in rf.classes_ else None

if hype_class_idx is None:
    print('⚠️ HYPE class not in training data. Rule-based fallback will be used.')
else:
    print(f'HYPE class is at index {hype_class_idx} in predict_proba output')

print('\nProbability breakdown per ticker:')
print('─' * 70)
header = f'{"Ticker":7s} | {" | ".join(f"{c:13s}" for c in class_names)} | Hype%'
print(header)
print('─' * 70)

for i, ticker in enumerate(tickers_list):
    prob_row = probas[i]
    prob_str = ' | '.join(f'{p:13.3f}' for p in prob_row)
    hype_pct = prob_row[hype_class_idx] * 100 if hype_class_idx is not None else 0
    icon     = '🔴' if hype_pct > 50 else '🟢'
    print(f'{icon} {ticker:6s} | {prob_str} | {hype_pct:.1f}%')

print('\n💡 The "Hype%" column is what drives the frontend gauge.')
print('   It flows: predict_proba() → hype_score → gauge needle position')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 12 — FINAL SUMMARY & HANDOFF CHECKLIST
# ═══════════════════════════════════════════════════════════

import os

files_to_check = {
    '../backend/models/random_forest.pkl':    'Main prediction model',
    '../backend/models/isolation_forest.pkl': 'Anomaly detector + scaler',
    '../data/training_data.csv':              'Raw training dataset',
    '../data/eda_feature_distributions.png':  'EDA chart for report',
    '../data/eda_feature_variance.png':       'Feature variance chart',
    '../data/eda_correlation_heatmap.png':    'Correlation heatmap',
    '../data/model_feature_importance.png':   'Feature importance chart',
}

print('=' * 55)
print('  NOTEBOOK 02 — HANDOFF CHECKLIST')
print('=' * 55)

all_ok = True
for path, desc in files_to_check.items():
    exists = os.path.exists(path)
    icon   = '✅' if exists else '❌'
    size   = f'({os.path.getsize(path)/1024:.1f} KB)' if exists else '(missing)'
    print(f'  {icon} {desc}')
    print(f'       {path} {size}')
    if not exists:
        all_ok = False

print()

# Quick model sanity check
test_input = np.zeros((1, len(FEATURES_FINAL)))
try:
    proba_test = rf.predict_proba(test_input)
    print(f'  ✅ rf.predict_proba(zeros) → shape {proba_test.shape} — OK')
except Exception as e:
    print(f'  ❌ rf.predict_proba() failed: {e}')
    all_ok = False

test_scaled = scaler.transform(test_input)
try:
    iso_test = iso.score_samples(test_scaled)
    print(f'  ✅ iso.score_samples(zeros) → {iso_test[0]:.3f} — OK')
except Exception as e:
    print(f'  ❌ iso.score_samples() failed: {e}')
    all_ok = False

print()
if all_ok:
    print('  ✅ ALL CHECKS PASSED. Give .pkl files to backend teammate.')
    print('  📁 Path: backend/models/random_forest.pkl')
    print('  📁 Path: backend/models/isolation_forest.pkl')
    print('\n  → Run Notebook 03_Validation.ipynb next.')
else:
    print('  ❌ Some checks failed. Fix before handing off.')